# PathSnap: Decision Tree on the existing preprocessed data

This notebook trains and evaluates a **Decision Tree** using the already-preprocessed PathSnap candidate data in `shared_data/`. It does not rerun the raw-data preprocessing.

Each candidate row has six numeric features and a binary target. A tree learns a sequence of feature-threshold questions (for example, whether the GPS-to-road distance is below a threshold) and estimates whether a candidate is positive. We use five-fold cross-validation grouped by trip, then fit a final tree on all training rows and evaluate on the student-test partition.

Unlike Logistic Regression and KNN, a Decision Tree does not need feature scaling. The parameters below match the earlier Decision Tree screening configuration so the new full-data experiment is easy to explain.


In [ ]:
from pathlib import Path
import json
import time
import numpy as np
from joblib import dump
from sklearn.metrics import f1_score, log_loss
from sklearn.model_selection import GroupKFold
from sklearn.tree import DecisionTreeClassifier

# Locate this portable bundle whether Jupyter starts here or at the repo root.
start = Path.cwd().resolve()
roots = [start, *start.parents]
bundle_candidates = [start / "DecisionTree"]
bundle_candidates.extend(p for p in roots if p.name == "DecisionTree")
BUNDLE = next(
    (p for p in bundle_candidates if (p / "04_decision_tree_same_preprocessed_data.ipynb").is_file()),
    None,
)
if BUNDLE is None:
    raise FileNotFoundError(
        "Could not locate the DecisionTree folder. Open this notebook from its folder or the repository root."
    )

required_files = [
    "train_candidates.npz", "train_metadata.json",
    "student_test_candidates.npz", "student_test_metadata.json",
]
data_candidates = [
    BUNDLE / "shared_data",
    BUNDLE.parent / "shared_data",
    BUNDLE.parent / "pathsnap" / "shared_data",
    BUNDLE.parent.parent / "pathsnap" / "shared_data",
]
DATA = next(
    (p for p in data_candidates if all((p / name).is_file() for name in required_files)),
    None,
)
if DATA is None:
    raise FileNotFoundError(
        "Preprocessed files not found. Copy the four train/student-test NPZ and metadata files into DecisionTree/shared_data/. See README.md."
    )
RESULTS = BUNDLE / "results"
ARTIFACTS = BUNDLE / "artifacts"
RESULTS.mkdir(exist_ok=True)
ARTIFACTS.mkdir(exist_ok=True)

train_file = np.load(DATA / "train_candidates.npz", allow_pickle=False)
test_file = np.load(DATA / "student_test_candidates.npz", allow_pickle=False)
train_meta = json.loads((DATA / "train_metadata.json").read_text())
test_meta = json.loads((DATA / "student_test_metadata.json").read_text())

X = train_file["X"].astype(np.float32, copy=False)
y = train_file["y"]
groups = train_file["trip_id"]
train_obs = train_file["gps_observation_id"]
X_test = test_file["X"].astype(np.float32, copy=False)
y_test = test_file["y"]
test_obs = test_file["gps_observation_id"]
feature_names = train_file["feature_names"].astype(str).tolist()

assert X.shape == (1_145_304, 6), f"Unexpected train data shape: {X.shape}"
assert X_test.shape == (450_648, 6), f"Unexpected student-test data shape: {X_test.shape}"
assert feature_names == train_meta["feature_names"]

print("Decision Tree bundle:", BUNDLE)
print("Preprocessed data:", DATA)
print("Train X:", X.shape, "Student-test X:", X_test.shape)
print("Features:", feature_names)
print("Train trips:", len(np.unique(groups)))
print("Train sampled GPS points:", train_meta["sampled_or_total_gps_points"])
print("Student-test GPS points:", test_meta["sampled_or_total_gps_points"])


## Decision Tree settings and evaluation helpers

- `max_depth=12` limits how many threshold questions can follow one another, reducing overfitting and keeping the tree interpretable.
- `min_samples_leaf=5` requires each leaf to represent at least five candidate rows.
- `class_weight="balanced"` compensates for the one-positive / 23-negative pattern among covered observations.
- `random_state=42` makes the split choices reproducible.

For each observation, we rank its candidates by predicted positive probability and check if the positive candidate is first. We also report candidate macro-F1 at a 0.5 threshold and log loss. Candidate misses are reported separately in the final result.


In [ ]:
PARAMS = {
    "max_depth": 12,
    "min_samples_leaf": 5,
    "class_weight": "balanced",
    "random_state": 42,
}
N_SPLITS = 5
BATCH_SIZE = 100_000


def make_model():
    return DecisionTreeClassifier(**PARAMS)


def predict_probabilities_batched(model, features, batch_size=BATCH_SIZE):
    pieces = []
    for start in range(0, len(features), batch_size):
        stop = min(start + batch_size, len(features))
        pieces.append(model.predict_proba(features[start:stop])[:, 1])
    return np.concatenate(pieces)


def observation_top1(y_true, probabilities, observation_ids):
    """Choose the highest-probability candidate per GPS observation."""
    _, inverse = np.unique(observation_ids, return_inverse=True)
    n_observations = int(inverse.max()) + 1
    best_probability = np.full(n_observations, -np.inf)
    np.maximum.at(best_probability, inverse, probabilities)
    is_winner = probabilities == best_probability[inverse]
    winning_rows = np.flatnonzero(is_winner)
    first_winner = np.full(n_observations, len(y_true), dtype=np.int64)
    np.minimum.at(first_winner, inverse[winning_rows], winning_rows)
    return float(np.mean(y_true[first_winner] == 1)), n_observations


def candidate_macro_f1(y_true, probabilities):
    return float(f1_score(y_true, probabilities >= 0.5, average="macro"))

print("Decision Tree parameters:", PARAMS)


## 1. Five-fold cross-validation by trip

All rows from a trip stay together in one fold. For each fold, the tree is trained on the other trips and scored on held-out trips. This guards against an overly optimistic score from related points in one trip appearing in both sets.

The actual per-fold scores and timings are saved to `results/decision_tree_cv.json`.


In [ ]:
cv = GroupKFold(n_splits=N_SPLITS)
fold_results = []
cv_started = time.perf_counter()

for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y, groups), start=1):
    print(f"Fold {fold}/{N_SPLITS}: fit rows={len(train_idx):,}; validation rows={len(valid_idx):,}", flush=True)
    model = make_model()
    fit_started = time.perf_counter()
    model.fit(X[train_idx], y[train_idx])
    fit_seconds = time.perf_counter() - fit_started

    predict_started = time.perf_counter()
    valid_probabilities = predict_probabilities_batched(model, X[valid_idx])
    predict_seconds = time.perf_counter() - predict_started
    top1, n_observations = observation_top1(y[valid_idx], valid_probabilities, train_obs[valid_idx])
    macro_f1 = candidate_macro_f1(y[valid_idx], valid_probabilities)
    fold_result = {
        "fold": fold,
        "held_out_trips": sorted(np.unique(groups[valid_idx]).astype(str).tolist()),
        "candidate_rows": int(len(valid_idx)),
        "observations": int(n_observations),
        "observation_top1": top1,
        "candidate_macro_f1_at_0.5": macro_f1,
        "fit_seconds": fit_seconds,
        "prediction_seconds": predict_seconds,
    }
    fold_results.append(fold_result)
    print(fold_result, flush=True)

cv_seconds = time.perf_counter() - cv_started
cv_summary = {
    "model": "DecisionTreeClassifier",
    "parameters": PARAMS,
    "features": feature_names,
    "train_rows": int(len(y)),
    "train_trips": int(len(np.unique(groups))),
    "cv_method": "5-fold GroupKFold by trip_id",
    "observation_top1_mean": float(np.mean([r["observation_top1"] for r in fold_results])),
    "observation_top1_std": float(np.std([r["observation_top1"] for r in fold_results])),
    "candidate_macro_f1_mean": float(np.mean([r["candidate_macro_f1_at_0.5"] for r in fold_results])),
    "folds": fold_results,
    "total_cv_seconds": cv_seconds,
}
(RESULTS / "decision_tree_cv.json").write_text(json.dumps(cv_summary, indent=2))
print("CV summary:")
print(json.dumps({k: v for k, v in cv_summary.items() if k != "folds"}, indent=2))


## 2. Fit the final tree and evaluate student test

After cross-validation, fit a fresh tree on all training candidate rows, save it, and score all candidate rows in the student-test table. This student-test partition is for development evaluation, not the reserved evaluator partition.


In [ ]:
final_model = make_model()
fit_started = time.perf_counter()
final_model.fit(X, y)
final_fit_seconds = time.perf_counter() - fit_started
dump(final_model, ARTIFACTS / "decision_tree.joblib")

infer_started = time.perf_counter()
test_probabilities = predict_probabilities_batched(final_model, X_test)
inference_seconds = time.perf_counter() - infer_started
student_top1, covered_observations = observation_top1(y_test, test_probabilities, test_obs)
student_macro_f1 = candidate_macro_f1(y_test, test_probabilities)
student_log_loss = float(log_loss(y_test, np.c_[1 - test_probabilities, test_probabilities], labels=[0, 1]))

results = {
    "model": "DecisionTreeClassifier",
    "parameters": PARAMS,
    "features": feature_names,
    "train_candidate_rows": int(len(y)),
    "train_gps_points": int(train_meta["sampled_or_total_gps_points"]),
    "train_candidate_miss_rate": float(train_meta["candidate_miss_rate"]),
    "student_test_candidate_rows": int(len(y_test)),
    "student_test_gps_points": int(test_meta["sampled_or_total_gps_points"]),
    "student_test_covered_points": int(covered_observations),
    "student_test_candidate_miss_rate": float(test_meta["candidate_miss_rate"]),
    "grouped_5fold_cv_observation_top1_mean": cv_summary["observation_top1_mean"],
    "grouped_5fold_cv_observation_top1_std": cv_summary["observation_top1_std"],
    "grouped_5fold_cv_candidate_macro_f1_mean": cv_summary["candidate_macro_f1_mean"],
    "student_test_observation_top1": student_top1,
    "student_test_candidate_macro_f1_at_0.5": student_macro_f1,
    "student_test_candidate_log_loss": student_log_loss,
    "final_fit_seconds": final_fit_seconds,
    "student_test_inference_seconds": inference_seconds,
    "artifact": "artifacts/decision_tree.joblib",
    "evaluation_note": "Top-1 is conditional on a positive reference candidate being generated. Labels are pointwise geometric pseudo-labels; these are candidate-ranking results, not sequence-level route accuracy.",
}
(RESULTS / "decision_tree_results.json").write_text(json.dumps(results, indent=2))
print(json.dumps(results, indent=2))


## Interpreting the output

The main “accuracy” number is `student_test_observation_top1`, which checks the top-ranked candidate for each covered GPS point. `student_test_candidate_macro_f1_at_0.5` is candidate-row classification quality; `student_test_candidate_log_loss` evaluates probability quality. `student_test_candidate_miss_rate` measures reference roads omitted by candidate generation and is not a Decision Tree error.

Run with the **Python (PathSnap local)** kernel and select **Run All**. The notebook reads the current shared preprocessed files and writes the model and JSON metrics separately from the RF, KNN, and Logistic Regression outputs.
